# Nível 1 — Parte A: Triagem de Operações PLD

Desafio técnico de estágio em IA para PLD (Prevenção à Lavagem de Dinheiro) — banco fictício.

**Objetivo desta parte:** carregar as operações de `dados/dados_nivel_1.json`, investigar a qualidade dos dados *antes* de qualquer tratamento, corrigir os problemas encontrados de forma documentada, normalizar valores para BRL, gerar agregações básicas e implementar duas regras de triagem (Regra 1 — Fracionamento; Regra 2 — Valor Atípico).

**Regra de ouro:** toda soma, contagem, mediana e comparação numérica é feita em pandas/Python puro — nenhum LLM é usado nesta etapa.

> Assume-se que o notebook é executado a partir da pasta `nivel_1/` (diretório padrão do Jupyter ao abrir este arquivo), por isso os caminhos de dados usam `../dados/`.

## 1. Carregamento dos dados

O JSON tem dois níveis: a taxa de câmbio (`taxa_cambio_usd_brl`) no topo e a lista de operações em `operacoes`. Carregamos o arquivo bruto com `json.load` para não perder a taxa de câmbio, e construímos o DataFrame apenas a partir da lista de operações.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_PATH = Path("..") / "dados" / "dados_nivel_1.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    dados_brutos = json.load(f)

taxa_cambio_usd_brl: float = dados_brutos["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados_brutos["operacoes"])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio_usd_brl}")
print(f"Formato do DataFrame (linhas, colunas): {df.shape}")
df.head(10)

Taxa de câmbio USD/BRL: 5.4
Formato do DataFrame (linhas, colunas): (20, 9)


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


## 2. Investigação de qualidade dos dados (antes de tratar)

Antes de corrigir qualquer coisa, precisamos saber exatamente o que está errado. Verificamos, nesta ordem:

1. **Duplicatas por `id`** — se o mesmo `id` aparece mais de uma vez, isso infla contagens e somas de forma silenciosa (grave para regras baseadas em soma/contagem, como a Regra 1).
2. **Valores nulos por coluna** — em especial `data`, que é usada para agrupar operações por dia.
3. **Tipos e valores únicos de `canal`, `tipo` e `moeda`** — para detectar inconsistências de categoria (typos, capitalização, categorias inesperadas) antes de usá-las em agregações.
4. **Outras anomalias** — valores não positivos, `dtypes` inesperados e contagem de clientes distintos.

In [2]:
# 2.1 Duplicatas por id
qtd_ids_duplicados = df.duplicated(subset="id").sum()
print(f"Linhas com id duplicado (excluindo a 1ª ocorrência): {qtd_ids_duplicados}")

print("\nTodas as linhas envolvidas em duplicidade de id:")
display(df[df.duplicated(subset="id", keep=False)].sort_values("id"))

# 2.2 Valores nulos por coluna
print("\nValores nulos por coluna:")
print(df.isna().sum())

# 2.3 Tipos e valores únicos de colunas categóricas
print("\ndtypes:")
print(df.dtypes)

print("\nValores únicos de 'canal':", sorted(df["canal"].dropna().unique().tolist()))
print("Valores únicos de 'tipo':", sorted(df["tipo"].dropna().unique().tolist()))
print("Valores únicos de 'moeda':", sorted(df["moeda"].dropna().unique().tolist()))

# 2.4 Outras anomalias
print("\nEstatísticas de 'valor':")
print(df["valor"].describe())

print("\nOperações com valor <= 0:", int((df["valor"] <= 0).sum()))
print(f"Clientes distintos ({df['cliente_id'].nunique()}):", sorted(df["cliente_id"].unique().tolist()))

Linhas com id duplicado (excluindo a 1ª ocorrência): 1

Todas as linhas envolvidas em duplicidade de id:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,



Valores nulos por coluna:
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

dtypes:
id               str
cliente_id       str
data             str
valor          int64
moeda            str
canal            str
tipo             str
contraparte      str
observacao       str
dtype: object

Valores únicos de 'canal': ['boleto', 'cartao', 'especie', 'pix', 'ted']
Valores únicos de 'tipo': ['deposito', 'pagamento', 'transferencia_enviada', 'transferencia_recebida']
Valores únicos de 'moeda': ['BRL', 'USD']

Estatísticas de 'valor':
count       20.000000
mean     11495.000000
std       7983.369227
min       1400.000000
25%       4175.000000
50%      10400.000000
75%      17225.000000
max      27000.000000
Name: valor, dtype: float64

Operações com valor <= 0: 0
Clientes distintos (6): ['CLI-A-1', 'CLI-A-2', 'CLI-A-3', 'CLI-A-4', 'CLI-A-5', 'CLI-A-6']


### Achados da investigação

- **1 `id` duplicado:** `OP-0007` (cliente `CLI-A-3`, 2026-03-05) aparece duas vezes com dados idênticos. Se não for removido antes de agrupar, infla artificialmente a soma diária desse cliente e poderia gerar um falso positivo na Regra 1.
- **1 `data` nula:** `OP-0017` (cliente `CLI-A-5`), com a observação `"data nao capturada pelo sistema"` — indica falha de captura, não ausência real de operação.
- **`canal`, `tipo` e `moeda` estão consistentes:** sem typos ou variação de capitalização. `canal` ∈ {pix, ted, boleto, cartao, especie}; `tipo` ∈ {transferencia_enviada, transferencia_recebida, pagamento, deposito}; `moeda` ∈ {BRL, USD}.
- **Sem valores não positivos** em `valor`.
- **6 clientes distintos**, como esperado (`CLI-A-1` a `CLI-A-6`).

Esses dois problemas (duplicata de `id` e `data` nula) são os únicos defeitos estruturais nos dados e serão tratados na próxima seção, cada um com sua justificativa.

## 3. Tratamento dos problemas encontrados

**3.1 Deduplicação por `id`.** Mantemos a primeira ocorrência (`keep="first"`) e descartamos as demais. Como as linhas duplicadas encontradas são idênticas em todos os campos, não há ambiguidade sobre qual "versão" manter — a decisão relevante é apenas não contar a mesma operação duas vezes nas somas e contagens.

**3.2 `data` nula.** Em vez de descartar a operação (ela é uma movimentação financeira real e precisa continuar auditável) ou inventar uma data, preenchemos o campo com o marcador textual `"DATA_PENDENTE"` e criamos a coluna booleana `ALERTA_DATA_AUSENTE` para tornar o problema visível e rastreável em qualquer relatório futuro — sem que ele se disfarce de dado válido.

**3.3 Regra de uso do marcador `DATA_PENDENTE`.** Uma data desconhecida não pode ser comparada com outras datas para saber se ocorreu "no mesmo dia" — por isso, operações com `DATA_PENDENTE`:
- são **excluídas do agrupamento diário da Regra 1** (Fracionamento), já que essa regra depende de saber se várias operações aconteceram na mesma data;
- **permanecem ativas** nas agregações gerais (volume por cliente, contagem por canal) e na Regra 2 (Valor Atípico), pois essas análises não dependem da data — excluir a operação delas esconderia volume financeiro real do cliente.

In [3]:
DATA_PENDENTE = "DATA_PENDENTE"

linhas_antes = len(df)

# 3.1 Deduplicação por id, preservando a primeira ocorrência
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)

print(f"Linhas antes da deduplicação: {linhas_antes}")
print(f"Linhas depois da deduplicação: {len(df)}")

# 3.2 Flag de alerta ANTES de preencher a data nula, para não perder a informação de ausência
df["ALERTA_DATA_AUSENTE"] = df["data"].isna()

# 3.3 Preenchimento do marcador de data pendente
df["data"] = df["data"].fillna(DATA_PENDENTE)

print(f"\nOperações com ALERTA_DATA_AUSENTE: {int(df['ALERTA_DATA_AUSENTE'].sum())}")
display(df[df["ALERTA_DATA_AUSENTE"]])

Linhas antes da deduplicação: 20
Linhas depois da deduplicação: 19

Operações com ALERTA_DATA_AUSENTE: 1


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,ALERTA_DATA_AUSENTE
16,OP-0017,CLI-A-5,DATA_PENDENTE,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema,True


## 4. Normalização de valores para BRL

Todas as regras de triagem comparam valores em uma única unidade monetária. Como `moeda` só assume `BRL` ou `USD` (confirmado na investigação), criamos `valor_brl`: quando `moeda == "USD"`, convertemos multiplicando por `taxa_cambio_usd_brl`; caso contrário, o valor já está em BRL e é mantido como está. A partir daqui, todas as agregações e regras usam `valor_brl`, nunca `valor`.

In [4]:
df["valor_brl"] = df["valor"].where(df["moeda"] != "USD", df["valor"] * taxa_cambio_usd_brl)

print("Operações em USD convertidas:")
display(df.loc[df["moeda"] == "USD", ["id", "cliente_id", "moeda", "valor", "valor_brl"]])

df[["id", "cliente_id", "moeda", "valor", "valor_brl"]].head(10)

Operações em USD convertidas:


,id,cliente_id,moeda,valor,valor_brl
12,OP-0013,CLI-A-4,USD,12000,64800.0


,id,cliente_id,moeda,valor,valor_brl
0,OP-0001,CLI-A-1,BRL,18100,18100.0
1,OP-0002,CLI-A-1,BRL,17300,17300.0
2,OP-0003,CLI-A-1,BRL,18800,18800.0
3,OP-0004,CLI-A-1,BRL,3300,3300.0
4,OP-0005,CLI-A-2,BRL,25900,25900.0
5,OP-0006,CLI-A-2,BRL,27000,27000.0
6,OP-0007,CLI-A-3,BRL,17200,17200.0
7,OP-0008,CLI-A-3,BRL,15200,15200.0
8,OP-0009,CLI-A-3,BRL,16100,16100.0
9,OP-0010,CLI-A-4,BRL,3800,3800.0


## 5. Agregações

Duas visões simples sobre os dados já tratados e normalizados:

- **Volume total transacionado por cliente** (`valor_brl`) — soma de todas as operações de cada cliente, incluindo as com `DATA_PENDENTE` (conforme decidido na Seção 3.3).
- **Quantidade de operações por canal** — contagem simples, útil para entender a distribuição de canais usados.

In [5]:
volume_por_cliente = (
    df.groupby("cliente_id")["valor_brl"]
    .sum()
    .sort_values(ascending=False)
    .rename("volume_total_brl")
)
print("Volume total transacionado por cliente (BRL):")
display(volume_por_cliente)

qtd_por_canal = df["canal"].value_counts().rename("quantidade_operacoes")
print("\nQuantidade de operações por canal:")
display(qtd_por_canal)

Volume total transacionado por cliente (BRL):


cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: volume_total_brl, dtype: float64


Quantidade de operações por canal:


canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: quantidade_operacoes, dtype: int64

## 6. Regra 1 — Fracionamento

**Critério:** sinaliza o **cliente** que, em uma **mesma data válida** (excluindo `DATA_PENDENTE` — ver Seção 3.3), realizou **3 ou mais operações** cuja **soma** de `valor_brl` **ultrapassa R$ 50.000,00**, **e** nenhuma operação isolada desse grupo **atinge R$ 20.000,00** (ou seja, todas ficam abaixo do limite individual). A condição de valor isolado existe justamente para capturar o padrão de fracionamento: várias operações pequenas que, somadas, superam um limite que uma única operação grande evitaria sozinha.

**Implementação em duas etapas:**
1. `identificar_grupos_fracionados` agrupa por `(cliente_id, data)` — apenas datas válidas — e aplica os três critérios (quantidade, soma, máximo individual) de forma vetorizada com pandas, retornando o conjunto de pares `(cliente_id, data)` que disparam a regra.
2. `aplicar_flag_fracionamento` usa esse conjunto para marcar, no DataFrame original, **todas as operações do grupo `(cliente_id, data)` flagrado** com `ALERTA_FRACIONAMENTO = True`. Optamos por marcar as operações do grupo (e não todo o histórico do cliente em outras datas) porque o padrão de fracionamento é definido pelo comportamento *naquele dia* — sinalizar operações não relacionadas, em datas sem indício algum, geraria ruído sem base na regra.

In [6]:
def identificar_grupos_fracionados(
    df: pd.DataFrame,
    limite_soma: float = 50_000.0,
    limite_operacao_isolada: float = 20_000.0,
    min_operacoes: int = 3,
    valor_pendente: str = DATA_PENDENTE,
) -> set[tuple[str, str]]:
    """Identifica pares (cliente_id, data) que caracterizam fracionamento (Regra 1).

    Um par (cliente, data) dispara a regra quando, considerando apenas
    operações com data válida (data != valor_pendente):
      1. o cliente tem `min_operacoes` ou mais operações naquela data;
      2. a soma de `valor_brl` dessas operações ultrapassa `limite_soma`; e
      3. nenhuma operação isolada do grupo atinge `limite_operacao_isolada`
         (i.e., o valor máximo do grupo é estritamente menor que o limite).

    Args:
        df: DataFrame de operações já tratado, contendo as colunas
            'cliente_id', 'data' e 'valor_brl'.
        limite_soma: valor que a soma diária precisa ultrapassar (exclusivo).
        limite_operacao_isolada: valor que nenhuma operação isolada pode atingir.
        min_operacoes: quantidade mínima de operações no mesmo dia.
        valor_pendente: marcador de data ausente, excluído do agrupamento.

    Returns:
        Conjunto de tuplas (cliente_id, data) que disparam a Regra 1.
    """
    operacoes_com_data_valida = df[df["data"] != valor_pendente]

    grupos = operacoes_com_data_valida.groupby(["cliente_id", "data"])["valor_brl"]
    quantidade = grupos.count()
    soma = grupos.sum()
    valor_maximo = grupos.max()

    disparado = (
        (quantidade >= min_operacoes)
        & (soma > limite_soma)
        & (valor_maximo < limite_operacao_isolada)
    )

    return set(disparado[disparado].index)


def aplicar_flag_fracionamento(
    df: pd.DataFrame, grupos_flagrados: set[tuple[str, str]]
) -> pd.Series:
    """Marca com True toda operação cujo par (cliente_id, data) está em `grupos_flagrados`."""
    pares_operacao = pd.Series(list(zip(df["cliente_id"], df["data"])), index=df.index)
    return pares_operacao.isin(grupos_flagrados)


grupos_fracionados = identificar_grupos_fracionados(df)
df["ALERTA_FRACIONAMENTO"] = aplicar_flag_fracionamento(df, grupos_fracionados)

print("Pares (cliente_id, data) flagrados pela Regra 1:", grupos_fracionados)
print(f"\nOperações sinalizadas: {int(df['ALERTA_FRACIONAMENTO'].sum())}")
display(df[df["ALERTA_FRACIONAMENTO"]][["id", "cliente_id", "data", "valor_brl"]])

Pares (cliente_id, data) flagrados pela Regra 1: {('CLI-A-1', '2026-03-09')}

Operações sinalizadas: 3


,id,cliente_id,data,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800.0


## 7. Regra 2 — Valor Atípico

**Critério:** sinaliza a **operação** cujo `valor_brl` é **superior a 5 vezes a mediana** dos valores (`valor_brl`) daquele mesmo cliente. A regra só é aplicada a **clientes com 4 ou mais operações** — com poucas operações a mediana é instável e qualquer valor um pouco maior pareceria "atípico" sem realmente ser.

**Por que mediana, e não média:** a mediana é robusta a outliers — a própria operação atípica não distorce o valor de referência tanto quanto distorceria uma média, tornando a comparação mais confiável.

**Universo considerado:** contagem de operações e mediana usam **todas** as operações do cliente, inclusive as com `DATA_PENDENTE` (Seção 3.3) — a data ausente não invalida o valor da operação para efeito desta regra, que não depende de data.

In [7]:
def identificar_valores_atipicos(
    df: pd.DataFrame,
    multiplicador_mediana: float = 5.0,
    min_operacoes_cliente: int = 4,
) -> pd.Series:
    """Identifica operações com valor atípico em relação ao próprio cliente (Regra 2).

    Uma operação é sinalizada quando:
      1. o cliente a que ela pertence tem `min_operacoes_cliente` ou mais
         operações no total (contando todas, inclusive com DATA_PENDENTE); e
      2. `valor_brl` da operação é estritamente maior que
         `multiplicador_mediana` vezes a mediana de `valor_brl` daquele cliente.

    Args:
        df: DataFrame de operações já tratado e normalizado, contendo as
            colunas 'cliente_id' e 'valor_brl'.
        multiplicador_mediana: quantas vezes a mediana o valor precisa superar.
        min_operacoes_cliente: quantidade mínima de operações do cliente para
            que ele entre na análise.

    Returns:
        pd.Series booleana alinhada ao índice de df, True nas operações atípicas.
    """
    operacoes_por_cliente = df.groupby("cliente_id")["id"].transform("count")
    mediana_por_cliente = df.groupby("cliente_id")["valor_brl"].transform("median")

    cliente_elegivel = operacoes_por_cliente >= min_operacoes_cliente
    limite_atipico = mediana_por_cliente * multiplicador_mediana

    return cliente_elegivel & (df["valor_brl"] > limite_atipico)


df["ALERTA_VALOR_ATIPICO"] = identificar_valores_atipicos(df)

print(f"Operações sinalizadas: {int(df['ALERTA_VALOR_ATIPICO'].sum())}")
display(df[df["ALERTA_VALOR_ATIPICO"]][["id", "cliente_id", "moeda", "valor", "valor_brl"]])

Operações sinalizadas: 1


,id,cliente_id,moeda,valor,valor_brl
12,OP-0013,CLI-A-4,USD,12000,64800.0


## 8. Validação explícita da Regra 1

Antes de seguir, comparamos lado a lado dois grupos que se parecem à primeira vista, mas que a Regra 1 precisa tratar de forma diferente:

- **Caso positivo (deve disparar):** `CLI-A-1`, 2026-03-09 — 3 operações (18.100 + 17.300 + 18.800 = 54.200), soma acima de R$ 50.000,00 e nenhuma operação isolada atinge R$ 20.000,00.
- **Caso negativo "parecido" (não deve disparar):** `CLI-A-2`, 2026-03-14 — 2 operações (25.900 + 27.000 = 52.900). A soma também ultrapassa R$ 50.000,00, mas o grupo tem apenas **2** operações — abaixo do mínimo de 3 exigido pela regra.

Se a função só checasse a soma, os dois casos disparariam igualmente — a validação abaixo prova que o critério de quantidade mínima é aplicado corretamente.

In [8]:
def resumir_grupo(df: pd.DataFrame, cliente_id: str, data: str, disparados: set[tuple[str, str]]) -> dict:
    """Resume um grupo (cliente_id, data) para fins de validação da Regra 1."""
    grupo = df[(df["cliente_id"] == cliente_id) & (df["data"] == data)]
    return {
        "cliente_id": cliente_id,
        "data": data,
        "qtd_operacoes": len(grupo),
        "soma_valor_brl": grupo["valor_brl"].sum(),
        "valor_max_isolado": grupo["valor_brl"].max(),
        "ALERTA_FRACIONAMENTO": (cliente_id, data) in disparados,
    }


# Recalculamos a partir da função da Seção 6 para que esta validação seja autocontida.
grupos_fracionados_validacao = identificar_grupos_fracionados(df)

caso_positivo = resumir_grupo(df, "CLI-A-1", "2026-03-09", grupos_fracionados_validacao)
caso_negativo = resumir_grupo(df, "CLI-A-2", "2026-03-14", grupos_fracionados_validacao)

tabela_validacao = pd.DataFrame(
    [
        {"caso": "Positivo (esperado: dispara)", **caso_positivo},
        {"caso": "Negativo parecido (esperado: não dispara)", **caso_negativo},
    ]
).set_index("caso")

display(tabela_validacao)

assert caso_positivo["ALERTA_FRACIONAMENTO"] is True, "Caso positivo deveria ter disparado a Regra 1"
assert caso_negativo["ALERTA_FRACIONAMENTO"] is False, "Caso negativo não deveria disparar a Regra 1"

print("Validação da Regra 1 (Fracionamento): os dois casos retornaram exatamente o resultado esperado.")

,cliente_id,data,qtd_operacoes,soma_valor_brl,valor_max_isolado,ALERTA_FRACIONAMENTO
caso,,,,,,
Positivo (esperado: dispara),CLI-A-1,2026-03-09,3,54200.0,18800.0,True
Negativo parecido (esperado: não dispara),CLI-A-2,2026-03-14,2,52900.0,27000.0,False


Validação da Regra 1 (Fracionamento): os dois casos retornaram exatamente o resultado esperado.


# Parte B — Parecer qualitativo com LLM (Gemini)

Na Parte A, todo cálculo (soma, contagem, mediana, comparação) foi feito em pandas — as duas regras de triagem já apontam **quem** e **por quê**. A Parte B usa um LLM (Gemini) apenas para o que pandas não faz bem: **interpretar** os dados já calculados e redigir um parecer qualitativo (tipologia suspeita, red flags, justificativa em linguagem natural). O LLM nunca recebe a tarefa de somar, contar ou comparar números — ele só recebe o resultado desses cálculos, prontos, e opina sobre eles.

## 9. Setup do cliente Gemini

Carregamos a chave de API do `.env` com `python-dotenv` — a chave nunca é escrita no código. Se a variável `GEMINI_API_KEY` não existir, o notebook falha imediatamente com uma mensagem clara, em vez de deixar o erro estourar de forma confusa lá na frente, na primeira chamada à API.

In [9]:
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY não encontrada nas variáveis de ambiente. "
        "Crie um arquivo .env na raiz do projeto com GEMINI_API_KEY=<sua_chave> "
        "(veja .env.example) antes de continuar."
    )

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")

client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Client Gemini inicializado com sucesso. Modelo configurado: {GEMINI_MODEL}")

Client Gemini inicializado com sucesso. Modelo configurado: gemini-3.6-flash


## 10. Seleção do cliente para análise

Primeiro listamos todo cliente sinalizado por **pelo menos uma** das duas regras (`ALERTA_FRACIONAMENTO` ou `ALERTA_VALOR_ATIPICO`), para escolher o caso de estudo a partir de um retrato completo — não da primeira linha que aparecer.

In [10]:
clientes_sinalizados = (
    df.groupby("cliente_id")[["ALERTA_FRACIONAMENTO", "ALERTA_VALOR_ATIPICO"]]
    .any()
    .assign(sinalizado=lambda x: x["ALERTA_FRACIONAMENTO"] | x["ALERTA_VALOR_ATIPICO"])
    .query("sinalizado")
    .drop(columns="sinalizado")
)

print("Clientes sinalizados por pelo menos uma regra:")
display(clientes_sinalizados)

Clientes sinalizados por pelo menos uma regra:


,ALERTA_FRACIONAMENTO,ALERTA_VALOR_ATIPICO
cliente_id,,
CLI-A-1,True,False
CLI-A-4,False,True


**Dois clientes sinalizados:** `CLI-A-1` (Regra 1 — Fracionamento) e `CLI-A-4` (Regra 2 — Valor Atípico).

**Escolha: `CLI-A-1`.** Motivos:
- É o caso de fracionamento — um padrão comportamental (3 operações no mesmo dia, mesma faixa de valor, todas abaixo do limite individual) que depende de contexto para ser bem interpretado, exatamente o tipo de análise em que um parecer qualitativo agrega mais valor sobre o que pandas já apontou.
- Tem múltiplos elementos para o LLM cruzar de forma qualitativa: dois canais diferentes (`pix` e `ted`) no mesmo dia, duas contrapartes distintas (`Alfa Comercio LTDA` e `Beta Servicos ME`), e uma quarta operação em outro canal (`boleto`) e outra data — material rico para observações sobre tipologia e red flags.
- `CLI-A-4` (Regra 2) é um caso mais direto — uma única remessa internacional em USD destoando das demais — e fica registrado aqui como alternativa válida, mas menos rico para ilustrar o valor do parecer qualitativo sobre um padrão comportamental.

In [11]:
CLIENTE_ANALISE = "CLI-A-1"

print(f"Cliente selecionado para o parecer com LLM: {CLIENTE_ANALISE}")
display(df[df["cliente_id"] == CLIENTE_ANALISE])

Cliente selecionado para o parecer com LLM: CLI-A-1


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,ALERTA_DATA_AUSENTE,valor_brl,ALERTA_FRACIONAMENTO,ALERTA_VALOR_ATIPICO
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,18100.0,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,17300.0,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False,18800.0,True,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,3300.0,False,False


## 11. Payload para o LLM

O payload carrega apenas dados **já calculados** em pandas — nunca valores brutos que exigiriam o LLM somar, contar ou comparar algo por conta própria:

- `quantidade_operacoes` e `volume_total_brl` — já somados/contados em pandas.
- `canais_utilizados` e `contrapartes` — já deduplicados e ordenados (`unique()`/`sorted()`), não listas brutas para o LLM filtrar.
- `flags_ativas` — o resultado *já pronto* das Regras 1 e 2 (e do alerta de data ausente), para o LLM saber exatamente quais regras dispararam sem precisar reavaliar critério algum.
- `historico_operacoes` — o detalhe operação a operação (data, valor já em BRL, canal, tipo, contraparte, observação), para o LLM ter contexto qualitativo (ex.: reconhecer múltiplas contrapartes no mesmo dia, ou uma observação suspeita) sem precisar recalcular nada.

O LLM recebe esse dicionário serializado como JSON e só precisa **interpretar** — nunca operar sobre — os números que já estão prontos.

In [12]:
def montar_payload_cliente(df: pd.DataFrame, cliente_id: str) -> dict:
    """Monta, a partir de dados já calculados em pandas, o payload de um cliente para o LLM.

    Nenhum valor no payload exige que o LLM some, conte ou compare números:
    tudo já vem agregado (volume, contagem) ou deduplicado (canais, contrapartes).

    Args:
        df: DataFrame de operações já tratado e normalizado (com valor_brl e flags).
        cliente_id: identificador do cliente a ser resumido.

    Returns:
        Dicionário serializável em JSON, pronto para compor o prompt do LLM.
    """
    operacoes_cliente = df[df["cliente_id"] == cliente_id]

    historico_operacoes = operacoes_cliente[
        ["id", "data", "valor_brl", "moeda", "canal", "tipo", "contraparte", "observacao"]
    ].to_dict(orient="records")

    return {
        "cliente_id": cliente_id,
        "quantidade_operacoes": int(len(operacoes_cliente)),
        "volume_total_brl": float(operacoes_cliente["valor_brl"].sum()),
        "canais_utilizados": sorted(operacoes_cliente["canal"].unique().tolist()),
        "contrapartes": sorted(operacoes_cliente["contraparte"].unique().tolist()),
        "flags_ativas": {
            "ALERTA_FRACIONAMENTO": bool(operacoes_cliente["ALERTA_FRACIONAMENTO"].any()),
            "ALERTA_VALOR_ATIPICO": bool(operacoes_cliente["ALERTA_VALOR_ATIPICO"].any()),
            "ALERTA_DATA_AUSENTE": bool(operacoes_cliente["ALERTA_DATA_AUSENTE"].any()),
        },
        "historico_operacoes": historico_operacoes,
    }


payload_cliente_analise = montar_payload_cliente(df, CLIENTE_ANALISE)

print(json.dumps(payload_cliente_analise, indent=2, ensure_ascii=False, sort_keys=True))

{
  "canais_utilizados": [
    "boleto",
    "pix",
    "ted"
  ],
  "cliente_id": "CLI-A-1",
  "contrapartes": [
    "Alfa Comercio LTDA",
    "Beta Servicos ME",
    "Gama Distribuidora"
  ],
  "flags_ativas": {
    "ALERTA_DATA_AUSENTE": false,
    "ALERTA_FRACIONAMENTO": true,
    "ALERTA_VALOR_ATIPICO": false
  },
  "historico_operacoes": [
    {
      "canal": "pix",
      "contraparte": "Alfa Comercio LTDA",
      "data": "2026-03-09",
      "id": "OP-0001",
      "moeda": "BRL",
      "observacao": "",
      "tipo": "transferencia_enviada",
      "valor_brl": 18100.0
    },
    {
      "canal": "pix",
      "contraparte": "Alfa Comercio LTDA",
      "data": "2026-03-09",
      "id": "OP-0002",
      "moeda": "BRL",
      "observacao": "",
      "tipo": "transferencia_enviada",
      "valor_brl": 17300.0
    },
    {
      "canal": "ted",
      "contraparte": "Beta Servicos ME",
      "data": "2026-03-09",
      "id": "OP-0003",
      "moeda": "BRL",
      "observacao": "",
    

## 12. Schema Pydantic do parecer (`ParecerPLD`)

O parecer do LLM é validado contra um schema Pydantic com cinco campos: `cliente_id`, `nivel_risco` (`Literal["baixo", "médio", "alto"]`), `tipologia_suspeita`, `red_flags` (lista) e `justificativa`.

**Por que normalizar acentuação antes de validar:** um LLM instruído a devolver `"médio"` pode, na prática, devolver `"medio"`, `"Médio"` ou `"MÉDIO"` — variações de acento e caixa que são semanticamente idênticas, mas que um `Literal` estrito rejeitaria como erro de validação. Em vez de tratar isso como falha (e queimar uma tentativa de retry por um detalhe cosmético), usamos um `field_validator` em modo `before` que remove acentos com `unicodedata.normalize("NFKD", ...)`, normaliza para minúsculas e mapeia de volta para o valor canônico (`"baixo"`, `"médio"` ou `"alto"`) antes do Pydantic validar contra o `Literal`. Um valor que não corresponda a nenhuma das três variantes continua sendo rejeitado — a normalização tolera *formatação*, não valores inválidos.

In [13]:
import unicodedata
from typing import Literal

from pydantic import BaseModel, Field, ValidationError, field_validator

NIVEIS_RISCO_VALIDOS: dict[str, str] = {"baixo": "baixo", "medio": "médio", "alto": "alto"}


def remover_acentos(texto: str) -> str:
    """Remove marcas de acentuação de uma string, preservando as letras base."""
    return "".join(
        caractere
        for caractere in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(caractere)
    )


class ParecerPLD(BaseModel):
    """Parecer qualitativo de PLD emitido pelo LLM para um cliente sinalizado."""

    cliente_id: str
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str] = Field(default_factory=list)
    justificativa: str

    @field_validator("nivel_risco", mode="before")
    @classmethod
    def normalizar_nivel_risco(cls, valor: object) -> object:
        """Normaliza acento/caixa de nivel_risco (ex.: 'MEDIO', 'Médio') para o valor canônico."""
        if not isinstance(valor, str):
            return valor

        chave_normalizada = remover_acentos(valor).strip().lower()

        if chave_normalizada not in NIVEIS_RISCO_VALIDOS:
            raise ValueError(
                f"nivel_risco {valor!r} não corresponde a 'baixo', 'médio' ou 'alto'."
            )

        return NIVEIS_RISCO_VALIDOS[chave_normalizada]


# Autoteste rápido da normalização, antes de qualquer chamada real ao LLM.
for variante in ["baixo", "MEDIO", "Médio", "médio", "ALTO", "Alto"]:
    parecer_teste = ParecerPLD(
        cliente_id="TESTE",
        nivel_risco=variante,
        tipologia_suspeita="teste",
        red_flags=[],
        justificativa="teste",
    )
    print(f"{variante!r:>10} -> {parecer_teste.nivel_risco!r}")

   'baixo' -> 'baixo'
   'MEDIO' -> 'médio'
   'Médio' -> 'médio'
   'médio' -> 'médio'
    'ALTO' -> 'alto'
    'Alto' -> 'alto'


## 13. Estratégia de cache (antes de qualquer chamada à API)

O cache é implementado **antes** da primeira chamada real ao LLM, por dois motivos práticos: (1) durante o desenvolvimento deste notebook, o mesmo prompt é reexecutado dezenas de vezes — sem cache, isso significa gastar cota da API e esperar latência de rede a cada `Run All`; (2) reprodutibilidade — reexecutar o notebook para revisão não deve gerar respostas diferentes nem custo adicional para o mesmo par (cliente, prompt, modelo).

A camada de cache tem quatro peças:
1. `chamar_llm_gemini` — a chamada real à API, sem cache, medindo tokens e latência. É a única função que efetivamente fala com o Gemini.
2. `gerar_chave_cache` — deriva uma chave MD5 determinística para identificar univocamente uma combinação (cliente, prompt, modelo, dados).
3. `salvar_cache` / `carregar_cache` — persistência em disco, um arquivo JSON por chave.
4. `consultar_llm_com_cache` — orquestra as três peças acima: cache hit devolve o que já foi salvo; cache miss chama a API e grava o resultado antes de devolver.

In [14]:
import hashlib
import time

from google.genai import types

CACHE_DIR = Path("..") / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


class ErroChamadaLLM(Exception):
    """Erro ao chamar a API do Gemini (rede, timeout, erro do serviço, etc.)."""


def chamar_llm_gemini(prompt: str, versao_modelo: str = GEMINI_MODEL) -> dict:
    """Faz uma chamada REAL (sem cache) à API do Gemini, medindo tokens e latência.

    Args:
        prompt: prompt completo a enviar ao modelo.
        versao_modelo: nome do modelo Gemini a usar.

    Returns:
        dict com 'resposta_bruta' (str, texto JSON devolvido pelo modelo),
        'tokens_prompt' (int), 'tokens_completion' (int) e 'latencia_segundos' (float).

    Raises:
        ErroChamadaLLM: se a chamada falhar por rede, timeout ou erro do serviço.
    """
    inicio = time.perf_counter()
    try:
        resposta = client.models.generate_content(
            model=versao_modelo,
            contents=prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json"),
        )
    except Exception as erro:
        raise ErroChamadaLLM(f"Falha ao chamar a API do Gemini: {erro}") from erro
    latencia_segundos = time.perf_counter() - inicio

    return {
        "resposta_bruta": resposta.text,
        "tokens_prompt": resposta.usage_metadata.prompt_token_count,
        "tokens_completion": resposta.usage_metadata.candidates_token_count,
        "latencia_segundos": latencia_segundos,
    }


def gerar_chave_cache(
    cliente_id: str,
    prompt_template: str,
    versao_modelo: str,
    dados_operacoes: list[dict],
) -> str:
    """Gera uma chave MD5 determinística para o cache de uma chamada ao LLM.

    A chave é derivada de cliente_id + hash MD5 do prompt_template + versao_modelo +
    um dump JSON (com chaves ordenadas) de dados_operacoes, de forma que a mesma
    combinação de entradas sempre produza a mesma chave, e qualquer mudança em
    qualquer uma delas produza uma chave diferente.

    Args:
        cliente_id: identificador do cliente analisado.
        prompt_template: texto do template de prompt usado (ex.: prompt V1 ou V2).
        versao_modelo: nome do modelo Gemini usado na chamada.
        dados_operacoes: lista de operações (dicts) que compõem o payload do cliente.

    Returns:
        Hash MD5 (string hexadecimal) que identifica essa combinação.
    """
    hash_prompt_template = hashlib.md5(prompt_template.encode("utf-8")).hexdigest()
    dump_dados_operacoes = json.dumps(dados_operacoes, sort_keys=True, ensure_ascii=False)

    material_chave = "::".join(
        [cliente_id, hash_prompt_template, versao_modelo, dump_dados_operacoes]
    )

    return hashlib.md5(material_chave.encode("utf-8")).hexdigest()


def _caminho_cache(chave: str) -> Path:
    return CACHE_DIR / f"{chave}.json"


def salvar_cache(chave: str, resultado: dict) -> None:
    """Salva em disco (.cache/<chave>.json) o resultado ORIGINAL de uma chamada ao LLM."""
    _caminho_cache(chave).write_text(
        json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def carregar_cache(chave: str) -> dict | None:
    """Carrega o resultado cacheado para uma chave, ou None se não existir cache."""
    caminho = _caminho_cache(chave)
    if not caminho.exists():
        return None
    return json.loads(caminho.read_text(encoding="utf-8"))


def consultar_llm_com_cache(
    prompt: str,
    cliente_id: str,
    prompt_template: str,
    versao_modelo: str,
    dados_operacoes: list[dict],
) -> dict:
    """Consulta o LLM com cache em disco: reaproveita uma resposta salva quando existe.

    Em caso de cache HIT, devolve exatamente o que foi salvo na chamada original
    (tokens e latência NÃO são recalculados). Em caso de cache MISS, faz a chamada
    real via `chamar_llm_gemini` e salva o resultado antes de devolvê-lo.

    Args:
        prompt: prompt completo já formatado a enviar ao modelo.
        cliente_id: identificador do cliente (usado para compor a chave de cache).
        prompt_template: texto do template do prompt (usado para compor a chave).
        versao_modelo: nome do modelo Gemini (usado para compor a chave).
        dados_operacoes: lista de operações do cliente (usada para compor a chave).

    Returns:
        dict com 'resposta_bruta', 'tokens_prompt', 'tokens_completion',
        'latencia_segundos', mais 'veio_do_cache' (bool) e 'chave_cache' (str).
    """
    chave = gerar_chave_cache(cliente_id, prompt_template, versao_modelo, dados_operacoes)

    resultado_cacheado = carregar_cache(chave)
    if resultado_cacheado is not None:
        print(f"[cache HIT] chave={chave} — resposta reaproveitada, nenhuma chamada nova à API.")
        return {**resultado_cacheado, "veio_do_cache": True, "chave_cache": chave}

    print(f"[cache MISS] chave={chave} — chamando a API do Gemini agora...")
    resultado = chamar_llm_gemini(prompt, versao_modelo=versao_modelo)
    salvar_cache(chave, resultado)

    return {**resultado, "veio_do_cache": False, "chave_cache": chave}


print(f"Diretório de cache: {CACHE_DIR.resolve()}")
print(f"Arquivos de cache existentes: {len(list(CACHE_DIR.glob('*.json')))}")

Diretório de cache: C:\Users\Samsung\triagem-pld-desafio\.cache
Arquivos de cache existentes: 4


### Por que `prompt_template` e `versao_modelo` precisam estar na chave

Se a chave de cache dependesse **só** de `cliente_id` + `dados_operacoes`, os prompts V1 e V2 (Seções 14 e 16) — que analisam o **mesmo** `CLIENTE_ANALISE` com o **mesmo** payload — colidiriam na mesma chave. Nesse cenário, a chamada do V1 preencheria o cache, e a "chamada" do V2 seria, na verdade, um cache HIT reaproveitando a resposta do V1 — nunca chegaria a consultar o modelo com o prompt de especialista. O teste A/B da Seção 17 estaria comparando a mesma resposta consigo mesma, e não duas respostas reais geradas por prompts diferentes.

Incluir o hash do `prompt_template` na chave resolve isso: V1 e V2 têm templates de texto diferentes, logo hashes diferentes, logo chaves diferentes — cada versão de prompt tem sua própria entrada de cache, independente da outra. `versao_modelo` entra pelo mesmo motivo, olhando para a frente: se um dia trocarmos de `gemini-2.0-flash` para outro modelo, não queremos reaproveitar por engano a resposta de um modelo diferente como se fosse a mesma.

## 14. Prompt V1 (zero-shot / direto)

Prompt direto, sem instruções adicionais de raciocínio, usado como linha de base para o teste A/B da Seção 17. O texto é montado exatamente como especificado, interpolando o JSON do payload (Seção 11) no lugar de `{dados_cliente_json}`:

> *"Você é um sistema de análise. Analise os dados do cliente {dados_cliente_json} e forneça um parecer com nivel_risco, tipologia_suspeita, red_flags e justificativa."*

A chamada passa por `consultar_llm_com_cache` (Seção 13) — na primeira execução deste notebook será um cache MISS (chamada real); em reexecuções seguintes, um cache HIT. `response_mime_type="application/json"` é fixado dentro de `chamar_llm_gemini`. O resultado é validado contra `ParecerPLD`; falha de validação é capturada com `try/except` e reportada, sem interromper o notebook.

In [15]:
PROMPT_TEMPLATE_V1 = (
    "Você é um sistema de análise. Analise os dados do cliente {dados_cliente_json} "
    "e forneça um parecer com nivel_risco, tipologia_suspeita, red_flags e justificativa."
)

dados_cliente_json = json.dumps(payload_cliente_analise, ensure_ascii=False)
prompt_v1 = PROMPT_TEMPLATE_V1.format(dados_cliente_json=dados_cliente_json)

parecer_v1 = None
try:
    resultado_v1 = consultar_llm_com_cache(
        prompt=prompt_v1,
        cliente_id=CLIENTE_ANALISE,
        prompt_template=PROMPT_TEMPLATE_V1,
        versao_modelo=GEMINI_MODEL,
        dados_operacoes=payload_cliente_analise["historico_operacoes"],
    )

    print(f"Veio do cache: {resultado_v1['veio_do_cache']}")
    print(f"Tokens prompt: {resultado_v1['tokens_prompt']} | Tokens completion: {resultado_v1['tokens_completion']}")
    print(f"Latência: {resultado_v1['latencia_segundos']:.3f}s")
    print("\nResposta bruta do modelo:")
    print(resultado_v1["resposta_bruta"])

    parecer_v1 = ParecerPLD.model_validate_json(resultado_v1["resposta_bruta"])
    print("\nParecer V1 validado com sucesso:")
    print(parecer_v1.model_dump_json(indent=2))
except ValidationError as erro_validacao:
    print("\nFalha de validação Pydantic no parecer V1 (JSON malformado ou fora do schema esperado):")
    print(erro_validacao)
except ErroChamadaLLM as erro_chamada:
    print("\nFalha ao chamar a API do Gemini para o parecer V1 (rede/timeout/serviço):")
    print(erro_chamada)
    resultado_v1 = None

[cache HIT] chave=b910ec530a4274530483a1b7fbbb469a — resposta reaproveitada, nenhuma chamada nova à API.
Veio do cache: True
Tokens prompt: 492 | Tokens completion: 341
Latência: 14.224s

Resposta bruta do modelo:
{
  "cliente_id": "CLI-A-1",
  "nivel_risco": "ALTO",
  "tipologia_suspeita": "Fracionamento de Operações (Structuring / Smurfing)",
  "red_flags": [
    "Múltiplas transferências de valor expressivo efetuadas no mesmo dia (09/03/2026)",
    "Fracionamento de valores individuais em patamares simétricos e próximos (R$ 18.100,00, R$ 17.300,00 e R$ 18.800,00)",
    "Concentração de 94% do volume total (R$ 54.200,00 de R$ 57.500,00) em uma única data via PIX e TED",
    "Alerta do sistema ativado para ALERTA_FRACIONAMENTO"
  ],
  "justificativa": "O cliente apresentou um comportamento transacional atípico no dia 09/03/2026, realizando três transferências de alto valor (duas para Alfa Comercio LTDA e uma para Beta Servicos ME) que somam R$ 54.200,00. A divisão dos montantes em ope

## 15. Estratégia de fallback para respostas malformadas

Uma resposta do LLM pode falhar por dois motivos distintos, e tratamos os dois: **erro de validação** (JSON malformado ou fora do schema — `ValidationError`) e **erro de chamada** (rede, timeout, erro do serviço — `ErroChamadaLLM`, já definida na Seção 13). `obter_parecer_com_fallback` é a função reutilizável que lida com ambos:

- **Até 3 tentativas.** A 1ª passa por `consultar_llm_com_cache` (elegível a cache, igual às Seções 14/16). Se falhar, as tentativas 2 e 3 chamam `chamar_llm_gemini` **diretamente, sem cache** — cachear uma resposta que já sabemos ser inválida não ajudaria, e cada retry deve ser uma nova chance real de o modelo acertar, não uma repetição do mesmo erro salvo em disco.
- **Backoff exponencial.** Antes de cada retry (tentativas 2 e 3), esperamos `2 ** tentativa` segundos (4s antes da 2ª, 8s antes da 3ª) — tempo para uma eventual instabilidade momentânea (rede, rate limit) se resolver sozinha.
- **Prompt reforçado no retry.** `formatar_prompt_retry` anexa ao prompt original um aviso explícito de que a tentativa anterior falhou e por quê, e reitera o formato exato esperado — dá ao modelo um sinal de correção, em vez de repetir o mesmo prompt que já falhou uma vez.
- **Fallback final: `ParecerPLDFallback`.** Se as 3 tentativas falharem, não forçamos o resultado no `Literal["baixo","médio","alto"]` de produção — isso mascararia uma falha do LLM como se fosse uma opinião real. Em vez disso, usamos um modelo Pydantic **separado**, com `nivel_risco: Literal["pendente_revisao_humana"]` e `tipologia_suspeita: Literal["FALHA_LLM_APOS_RETRIES"]`, tornando o caso de falha estruturalmente distinguível de um parecer real (`ParecerPLD | ParecerPLDFallback`). A `justificativa` desse parecer de contingência não vem do LLM (ele falhou) — vem do resumo agregado que **já foi calculado em pandas** para o cliente (Seção 11), garantindo que mesmo na pior hipótese o analista humano recebe algo confiável e útil.

In [16]:
class ParecerPLDFallback(BaseModel):
    """Parecer de contingência quando o LLM falha em produzir uma resposta válida.

    Deliberadamente um schema separado de ParecerPLD: nivel_risco e
    tipologia_suspeita são Literals de valor único, para que uma falha do LLM
    nunca seja confundida com um parecer de produção real.
    """

    cliente_id: str
    nivel_risco: Literal["pendente_revisao_humana"]
    tipologia_suspeita: Literal["FALHA_LLM_APOS_RETRIES"]
    red_flags: list[str] = Field(default_factory=list)
    justificativa: str


def resumir_cliente_para_fallback(payload_cliente: dict, tentativas_realizadas: int) -> str:
    """Monta a justificativa do parecer de fallback a partir do resumo já calculado em pandas."""
    flags_ativas = [nome for nome, ativo in payload_cliente["flags_ativas"].items() if ativo]
    flags_texto = ", ".join(flags_ativas) if flags_ativas else "nenhuma"

    return (
        "[Resumo agregado calculado em pandas — o texto abaixo não veio do LLM] "
        f"Cliente {payload_cliente['cliente_id']}: {payload_cliente['quantidade_operacoes']} "
        f"operação(ões), volume total de R$ {payload_cliente['volume_total_brl']:.2f}, "
        f"canais utilizados: {', '.join(payload_cliente['canais_utilizados'])}, "
        f"contrapartes: {', '.join(payload_cliente['contrapartes'])}, "
        f"flags ativas: {flags_texto}. "
        f"O LLM não produziu um parecer válido após {tentativas_realizadas} tentativas — "
        "este caso requer revisão humana manual."
    )


def formatar_prompt_retry(prompt_original: str, numero_tentativa: int, motivo_falha: str) -> str:
    """Reforça o prompt original com um aviso explícito de que a tentativa anterior falhou."""
    return (
        f"{prompt_original}\n\n"
        f"ATENÇÃO: a tentativa anterior (nº {numero_tentativa - 1}) falhou pelo seguinte motivo: "
        f"{motivo_falha}\n"
        "Responda novamente. A resposta DEVE ser um único objeto JSON válido, sem nenhum "
        "texto fora do JSON, contendo exatamente os campos: cliente_id (string), nivel_risco "
        "('baixo', 'médio' ou 'alto'), tipologia_suspeita (string), red_flags (lista de strings) "
        "e justificativa (string)."
    )


def obter_parecer_com_fallback(
    prompt_formatado: str,
    prompt_template: str,
    cliente_id: str,
    versao_modelo: str,
    dados_operacoes: list[dict],
    payload_cliente: dict,
    max_tentativas: int = 3,
) -> tuple[ParecerPLD | ParecerPLDFallback, dict]:
    """Obtém um parecer PLD com até `max_tentativas`, backoff exponencial e fallback final.

    A 1ª tentativa passa pelo cache (`consultar_llm_com_cache`). As tentativas
    seguintes são chamadas reais sem cache (`chamar_llm_gemini`), precedidas de
    um backoff de 2**tentativa segundos e de um prompt reforçado indicando que a
    tentativa anterior falhou. Se todas falharem, devolve um `ParecerPLDFallback`
    com justificativa baseada no resumo já calculado em pandas para o cliente.

    Args:
        prompt_formatado: prompt já pronto (com o payload interpolado) para a 1ª tentativa.
        prompt_template: template de prompt (sem interpolação), usado na chave de cache.
        cliente_id: identificador do cliente analisado.
        versao_modelo: nome do modelo Gemini.
        dados_operacoes: histórico de operações do cliente (para a chave de cache).
        payload_cliente: payload completo do cliente (para montar a justificativa de fallback).
        max_tentativas: número máximo de tentativas antes de desistir.

    Returns:
        Tupla (parecer, resultado_da_chamada), onde parecer é um ParecerPLD (sucesso)
        ou ParecerPLDFallback (falha após todas as tentativas).
    """
    ultimo_erro: Exception | None = None
    ultimo_resultado: dict | None = None

    for tentativa in range(1, max_tentativas + 1):
        try:
            if tentativa == 1:
                resultado = consultar_llm_com_cache(
                    prompt=prompt_formatado,
                    cliente_id=cliente_id,
                    prompt_template=prompt_template,
                    versao_modelo=versao_modelo,
                    dados_operacoes=dados_operacoes,
                )
            else:
                espera_segundos = 2**tentativa
                print(
                    f"[fallback] aguardando {espera_segundos}s (backoff exponencial) "
                    f"antes da tentativa {tentativa}/{max_tentativas}..."
                )
                time.sleep(espera_segundos)

                prompt_retry = formatar_prompt_retry(prompt_formatado, tentativa, str(ultimo_erro))
                print(f"[fallback] tentativa {tentativa}/{max_tentativas} — chamada real, SEM cache.")
                resultado_bruto = chamar_llm_gemini(prompt_retry, versao_modelo=versao_modelo)
                resultado = {**resultado_bruto, "veio_do_cache": False, "chave_cache": None}

            ultimo_resultado = resultado
            parecer = ParecerPLD.model_validate_json(resultado["resposta_bruta"])
            print(f"[fallback] tentativa {tentativa}/{max_tentativas} validada com sucesso.")
            return parecer, resultado

        except (ValidationError, ErroChamadaLLM) as erro:
            ultimo_erro = erro
            print(f"[fallback] tentativa {tentativa}/{max_tentativas} falhou: {erro}")

    print(f"[fallback] todas as {max_tentativas} tentativas falharam — gerando parecer de contingência.")
    parecer_fallback = ParecerPLDFallback(
        cliente_id=cliente_id,
        nivel_risco="pendente_revisao_humana",
        tipologia_suspeita="FALHA_LLM_APOS_RETRIES",
        red_flags=[],
        justificativa=resumir_cliente_para_fallback(payload_cliente, max_tentativas),
    )
    resultado_fallback = {
        "resposta_bruta": parecer_fallback.model_dump_json(),
        "tokens_prompt": (ultimo_resultado or {}).get("tokens_prompt"),
        "tokens_completion": (ultimo_resultado or {}).get("tokens_completion"),
        "latencia_segundos": (ultimo_resultado or {}).get("latencia_segundos"),
        "veio_do_cache": False,
        "chave_cache": None,
    }
    return parecer_fallback, resultado_fallback


print("Fallback definido: obter_parecer_com_fallback, ParecerPLDFallback, formatar_prompt_retry, "
      "resumir_cliente_para_fallback.")

Fallback definido: obter_parecer_com_fallback, ParecerPLDFallback, formatar_prompt_retry, resumir_cliente_para_fallback.

### Autoteste do fallback (sem gastar cota real de API)

Forçar 3 falhas reais e consecutivas do Gemini para provar o mecanismo não é prático nem determinístico. Em vez disso, isolamos `chamar_llm_gemini` com `unittest.mock.patch` para sempre devolver uma resposta malformada, e verificamos que `obter_parecer_com_fallback` esgota as 3 tentativas (respeitando o backoff) e devolve exatamente o `ParecerPLDFallback` esperado, com a justificativa vinda do resumo em pandas — não do LLM. Esta célula não faz nenhuma chamada real à API; leva ~12s só pelo backoff (4s + 8s) sendo aguardado de verdade.

In [17]:
from unittest.mock import patch


def _resposta_falsa_malformada(prompt: str, versao_modelo: str = GEMINI_MODEL) -> dict:
    """Simula chamar_llm_gemini sempre devolvendo um JSON inválido, para testar o fallback."""
    return {
        "resposta_bruta": "isto não é um JSON válido",
        "tokens_prompt": 10,
        "tokens_completion": 5,
        "latencia_segundos": 0.01,
    }


with patch(f"{__name__}.chamar_llm_gemini", side_effect=_resposta_falsa_malformada):
    parecer_teste, resultado_teste = obter_parecer_com_fallback(
        prompt_formatado="[AUTOTESTE] prompt fictício para forçar falha do fallback",
        prompt_template="[AUTOTESTE] template fictício — nunca usado pelos prompts V1/V2 reais",
        cliente_id=CLIENTE_ANALISE,
        versao_modelo=GEMINI_MODEL,
        dados_operacoes=payload_cliente_analise["historico_operacoes"],
        payload_cliente=payload_cliente_analise,
    )

print("\nParecer resultante do autoteste:")
print(parecer_teste.model_dump_json(indent=2))

assert isinstance(parecer_teste, ParecerPLDFallback), "Esperava um ParecerPLDFallback após 3 falhas simuladas"
assert parecer_teste.nivel_risco == "pendente_revisao_humana"
assert parecer_teste.tipologia_suspeita == "FALHA_LLM_APOS_RETRIES"
assert "pandas" in parecer_teste.justificativa
assert resultado_teste["veio_do_cache"] is False

print("\nAutoteste do fallback: OK — 3 tentativas falharam (proposital) e o parecer de contingência foi gerado corretamente.")

[cache HIT] chave=4daa1e7262a83ac915cf00f023faa12a — resposta reaproveitada, nenhuma chamada nova à API.
[fallback] tentativa 1/3 falhou: 1 validation error for ParecerPLD
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='isto não é um JSON válido', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
[fallback] aguardando 4s (backoff exponencial) antes da tentativa 2/3...


[fallback] tentativa 2/3 — chamada real, SEM cache.
[fallback] tentativa 2/3 falhou: 1 validation error for ParecerPLD
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='isto não é um JSON válido', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
[fallback] aguardando 8s (backoff exponencial) antes da tentativa 3/3...


[fallback] tentativa 3/3 — chamada real, SEM cache.
[fallback] tentativa 3/3 falhou: 1 validation error for ParecerPLD
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='isto não é um JSON válido', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
[fallback] todas as 3 tentativas falharam — gerando parecer de contingência.

Parecer resultante do autoteste:
{
  "cliente_id": "CLI-A-1",
  "nivel_risco": "pendente_revisao_humana",
  "tipologia_suspeita": "FALHA_LLM_APOS_RETRIES",
  "red_flags": [],
  "justificativa": "[Resumo agregado calculado em pandas — o texto abaixo não veio do LLM] Cliente CLI-A-1: 4 operação(ões), volume total de R$ 57500.00, canais utilizados: boleto, pix, ted, contrapartes: Alfa Comercio LTDA, Beta Servicos ME, Gama Distribuidora, flags ativas: ALERTA_FRACIONAMENTO. O LLM não produziu um parecer válido após 3 tentativas — este caso requer revisão humana manual."
}

Autoteste do fallba

## 16. Prompt V2 (especialista / chain-of-thought)

Mesmo cliente, mesmo payload, mesma função de cache — só o `prompt_template` muda, o que basta para gerar uma chave de cache diferente da V1 (Seção 13). O prompt de especialista pede explicitamente para não recalcular nada e foca o LLM no que ele faz de melhor — julgamento qualitativo sobre coerência de canal, observações e falhas de governança:

> *"Você é um Auditor Sênior de PLD. NÃO REFAÇA CÁLCULOS MATEMÁTICOS. Foque no aspecto qualitativo: coerência dos canais (ex.: espécie), análise das observações e falhas de governança (ex.: DATA_PENDENTE). Analise {dados_cliente_json} e devolva a estrutura JSON validada."*

Repetimos exatamente o mesmo processo da Seção 14: chamada via `consultar_llm_com_cache`, registro de tokens/latência (do cache ou de uma chamada real), validação contra `ParecerPLD` com `try/except`.

In [18]:
PROMPT_TEMPLATE_V2 = (
    "Você é um Auditor Sênior de PLD. NÃO REFAÇA CÁLCULOS MATEMÁTICOS. Foque no aspecto "
    "qualitativo: coerência dos canais (ex.: espécie), análise das observações e falhas de "
    "governança (ex.: DATA_PENDENTE). Analise {dados_cliente_json} e devolva a estrutura JSON "
    "validada."
)

prompt_v2 = PROMPT_TEMPLATE_V2.format(dados_cliente_json=dados_cliente_json)

parecer_v2 = None
try:
    resultado_v2 = consultar_llm_com_cache(
        prompt=prompt_v2,
        cliente_id=CLIENTE_ANALISE,
        prompt_template=PROMPT_TEMPLATE_V2,
        versao_modelo=GEMINI_MODEL,
        dados_operacoes=payload_cliente_analise["historico_operacoes"],
    )

    print(f"Veio do cache: {resultado_v2['veio_do_cache']}")
    print(f"Tokens prompt: {resultado_v2['tokens_prompt']} | Tokens completion: {resultado_v2['tokens_completion']}")
    print(f"Latência: {resultado_v2['latencia_segundos']:.3f}s")
    print("\nResposta bruta do modelo:")
    print(resultado_v2["resposta_bruta"])

    parecer_v2 = ParecerPLD.model_validate_json(resultado_v2["resposta_bruta"])
    print("\nParecer V2 validado com sucesso:")
    print(parecer_v2.model_dump_json(indent=2))
except ValidationError as erro_validacao:
    print("\nFalha de validação Pydantic no parecer V2 (JSON malformado ou fora do schema esperado):")
    print(erro_validacao)
except ErroChamadaLLM as erro_chamada:
    print("\nFalha ao chamar a API do Gemini para o parecer V2 (rede/timeout/serviço):")
    print(erro_chamada)
    resultado_v2 = None

if resultado_v1 is not None and resultado_v2 is not None:
    print(f"\nChave de cache V1: {resultado_v1['chave_cache']}")
    print(f"Chave de cache V2: {resultado_v2['chave_cache']}")
    assert resultado_v1["chave_cache"] != resultado_v2["chave_cache"], (
        "V1 e V2 geraram a MESMA chave de cache — a correção da Seção 13 falhou."
    )
    print("Chaves diferentes confirmadas: V1 e V2 não colidem no cache.")

[cache HIT] chave=80e6b81d2f0a45f7b674f865bfee5a51 — resposta reaproveitada, nenhuma chamada nova à API.
Veio do cache: True
Tokens prompt: 526 | Tokens completion: 603
Latência: 27.090s

Resposta bruta do modelo:
{
  "parecer_auditoria": {
    "cliente_id": "CLI-A-1",
    "classificacao_risco": "ALTO",
    "resumo_analitico": "Identificado padrão consistente com fracionamento intradiário de operações (smurfing) no dia 09/03/2026, visando possivelmente burlar alçadas ou limites de monitoramento automatizado. Há déficit de governança na coleta de dados contextuais das transações.",
    "analise_qualitativa": {
      "coerencia_canais_e_velocidade": "A utilização de PIX e TED em um mesmo dia com valores expressivos e fracionados indica alta velocidade de transposição de recursos para terceiros (Alfa Comercio LTDA e Beta Servicos ME). Ausência de transações em espécie atenua o risco de origem física, mas o uso combinado de PIX/TED no mesmo dia para a mesma contraparte eleva o risco compor

## 17. Comparação final — V1 vs. V2

Colocamos as duas respostas lado a lado: tokens, latência, se vieram do cache, e o parecer completo de cada uma. As chaves de cache (`chave_cache`) são comparadas explicitamente — precisam ser diferentes, senão V1 e V2 teriam colidido e uma delas seria apenas uma cópia da outra reaproveitada do cache, invalidando o teste A/B (motivo detalhado na Seção 13).

In [19]:
if resultado_v1 is None or resultado_v2 is None:
    raise RuntimeError(
        "V1 e/ou V2 não completaram com sucesso (veja os erros impressos nas Seções 14/16) — "
        "não há o que comparar."
    )


def parecer_para_linha(versao: str, resultado: dict, parecer: ParecerPLD | ParecerPLDFallback | None) -> dict:
    linha = {
        "versao": versao,
        "veio_do_cache": resultado["veio_do_cache"],
        "chave_cache": resultado["chave_cache"],
        "tokens_prompt": resultado["tokens_prompt"],
        "tokens_completion": resultado["tokens_completion"],
        "latencia_segundos": resultado["latencia_segundos"],
    }
    if parecer is not None:
        linha.update(
            {
                "nivel_risco": parecer.nivel_risco,
                "tipologia_suspeita": parecer.tipologia_suspeita,
                "red_flags": parecer.red_flags,
                "justificativa": parecer.justificativa,
            }
        )
    return linha


tabela_comparacao = pd.DataFrame(
    [
        parecer_para_linha("V1 (zero-shot)", resultado_v1, parecer_v1),
        parecer_para_linha("V2 (especialista)", resultado_v2, parecer_v2),
    ]
).set_index("versao")

print("Resumo comparativo (tokens, latência, cache, nível de risco):")
display(tabela_comparacao[["veio_do_cache", "chave_cache", "tokens_prompt", "tokens_completion", "latencia_segundos", "nivel_risco"]])

for versao, resultado, parecer in [
    ("V1 (zero-shot)", resultado_v1, parecer_v1),
    ("V2 (especialista)", resultado_v2, parecer_v2),
]:
    print(f"\n=== {versao} ===")
    print(
        f"veio_do_cache={resultado['veio_do_cache']} | tokens_prompt={resultado['tokens_prompt']} | "
        f"tokens_completion={resultado['tokens_completion']} | latencia={resultado['latencia_segundos']:.3f}s"
    )
    if parecer is not None:
        print(parecer.model_dump_json(indent=2))
    else:
        print("(parecer não pôde ser validado)")

assert resultado_v1["chave_cache"] != resultado_v2["chave_cache"], (
    "V1 e V2 compartilham a mesma chave de cache — o teste A/B não é válido."
)
print(
    "\nConfirmado: V1 e V2 têm chaves de cache diferentes — as duas respostas acima vieram de "
    "chamadas reais e independentes, não de um cache compartilhado indevidamente."
)

Resumo comparativo (tokens, latência, cache, nível de risco):


,veio_do_cache,chave_cache,tokens_prompt,tokens_completion,latencia_segundos,nivel_risco
versao,,,,,,
V1 (zero-shot),True,b910ec530a4274530483a1b7fbbb469a,492,341,14.223743,alto
V2 (especialista),True,80e6b81d2f0a45f7b674f865bfee5a51,526,603,27.090225,NaN



=== V1 (zero-shot) ===
veio_do_cache=True | tokens_prompt=492 | tokens_completion=341 | latencia=14.224s
{
  "cliente_id": "CLI-A-1",
  "nivel_risco": "alto",
  "tipologia_suspeita": "Fracionamento de Operações (Structuring / Smurfing)",
  "red_flags": [
    "Múltiplas transferências de valor expressivo efetuadas no mesmo dia (09/03/2026)",
    "Fracionamento de valores individuais em patamares simétricos e próximos (R$ 18.100,00, R$ 17.300,00 e R$ 18.800,00)",
    "Concentração de 94% do volume total (R$ 54.200,00 de R$ 57.500,00) em uma única data via PIX e TED",
    "Alerta do sistema ativado para ALERTA_FRACIONAMENTO"
  ],
  "justificativa": "O cliente apresentou um comportamento transacional atípico no dia 09/03/2026, realizando três transferências de alto valor (duas para Alfa Comercio LTDA e uma para Beta Servicos ME) que somam R$ 54.200,00. A divisão dos montantes em operações sucessivas na mesma data sugere uma estratégia intencional de fracionamento (smurfing) para burlar li

### Diferença qualitativa observada entre V1 e V2

Com `CLIENTE_ANALISE = "CLI-A-1"` e `gemini-3.6-flash`, as duas versões divergiram de um jeito instrutivo:

- **V1 validou de primeira; V2 falhou a validação Pydantic.** V1 devolveu exatamente os quatro campos pedidos (`nivel_risco`, `tipologia_suspeita`, `red_flags`, `justificativa`) e passou pelo schema sem erro. V2 devolveu uma resposta bem mais elaborada, mas **aninhada** sob uma chave própria (`"parecer_auditoria"`), com campos que o modelo inventou (`classificacao_risco`, `resumo_analitico`, `falhas_governanca_e_processo`, `recomendações_pld`...) em vez dos campos pedidos — o `try/except` da Seção 16 capturou isso como `ValidationError` exatamente como projetado, sem derrubar o notebook. A hipótese mais provável: o prompt V1 lista os quatro nomes de campo explicitamente no texto ("...forneça um parecer com nivel_risco, tipologia_suspeita, red_flags e justificativa"), enquanto o V2 pede apenas "devolva a estrutura JSON validada" — sem repetir os nomes dos campos —, e a persona de "Auditor Sênior" parece ter empurrado o modelo para o formato de um relatório de auditoria mais rico, e não para o schema plano que o Pydantic exige. Isso reforça, na prática, por que a Seção 15 (fallback/retry) existe: um prompt bem escrito no conteúdo pode ainda assim falhar na forma.
- **V2 é qualitativamente mais rico no que foi pedido.** Mesmo inválido estruturalmente, o conteúdo do V2 endereça muito mais de perto o que o prompt de especialista pediu: comenta a ausência de operações em espécie (reduzindo um vetor de risco), destaca que **todas as observações vieram vazias** (achado que o V1 não menciona), e vai além do parecer básico com `falhas_governanca_e_processo` e uma recomendação concreta de escalonamento ao Comitê de PLD/COAF. V1 é mais direto e genérico, focado quase inteiramente no padrão numérico do fracionamento.
- **Um limite real da "regra de ouro" aparece no próprio V1.** Mesmo sem instrução para tal (o prompt V1 não pede "não calcule"), o red flag do V1 cita "94% do volume total (R$ 54.200,00 de R$ 57.500,00)" — um percentual que **não estava no payload**: o modelo somou as três operações do dia e dividiu pelo `volume_total_brl` por conta própria. É um lembrete de que a instrução "não recalcule" (só presente no prompt V2) importa, e de que nenhuma decisão de negócio deste notebook depende desse número — ele aparece só na prosa do parecer, nunca em um campo estruturado usado por Regra 1/Regra 2.
- **Custo:** V2 usou bem mais tokens de saída (603 vs. 341) e teve latência maior (27,1s vs. 14,2s) para produzir uma resposta mais longa — mesmo que inválida contra o schema.